# AFM Force-Curve Analysis — Contact Point Detection & Cantilever Linearity

This notebook walks through three progressively better contact-point detection
methods, explains *why* each one fails or succeeds, and shows how to extract
INVOLS from the cantilever's linear operating range.

**Test data**: two force curves with long, sloped baselines and short contact
regions (~3–7 % of the curve). These are worst-case for classical methods.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from io_utils import (
    FCConfig, read_lvm_raw, detect_turnaround, split_at_turnaround,
)
from contact_point import (
    estimate_contact_approach, estimate_contact_retract,
    find_contact_piecewise, _baseline_deviation, _gradient_cp,
    _residual_two_lines,
    fit_contact_region, compute_phase_angle,
    ContactResult,
)

plt.rcParams.update({
    "figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.3,
    "font.size": 10, "axes.titleweight": "bold",
})
C_APP, C_RET = "cornflowerblue", "orange"
C_FIT, C_FIT_R = "#2255aa", "#cc7700" 

## 1. Data Loading

Each `.lvm` file contains four blocks of $N$ samples (deflection approach,
deflection retract, Z approach, Z retract). We concatenate both halves and
split at the physical turnaround $\arg\max(z)$.

In [ ]:
# Adjust num_app / num_ret to match your files.
# 48 000 lines → 12 000 per channel; 120 000 lines → 30 000 per channel.
num_app = 12000
num_ret = 12000

curves = {}
for fname in ["ForceCurve_00008.lvm", "ForceCurve_00010.lvm"]:
    raw = read_lvm_raw(Path(fname), num_app, num_ret)
    turn = detect_turnaround(raw)
    fc = split_at_turnaround(raw, turn)
    curves[fname] = fc
    n_a = len(fc.app_z_V)
    print(f"{fname}: approach {n_a} pts, retract {len(fc.ret_z_V)} pts, "
          f"turnaround at {turn}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
for ax, (fname, fc) in zip(axes, curves.items()):
    ax.plot(fc.app_z_V, fc.app_defl_V, color=C_APP, lw=0.5, alpha=0.7,
            label="Approach")
    ax.plot(fc.ret_z_V, fc.ret_defl_V, color=C_RET, lw=0.5, alpha=0.7,
            label="Retract")
    ax.set_xlabel("Z (V)"); ax.set_ylabel("Deflection (V)")
    ax.set_title(fname, fontsize=10); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 2. The Baseline Problem

Before we can detect the contact point, let's understand the baseline.
Ideally it's flat, but in practice the baseline often has a **slope** due
to optical interference, laser drift, or cantilever bowing.

We fit a line to the first 10 % and measure how well it extrapolates.

In [ ]:
for fname, fc in curves.items():
    z, d = fc.app_z_V, fc.app_defl_V
    n = len(z)
    bl_end = int(n * 0.10)

    # Fit baseline
    coeffs = np.polyfit(z[:bl_end], d[:bl_end], 1)
    bl_pred = np.polyval(coeffs, z)
    noise = np.std(d[:bl_end] - bl_pred[:bl_end])
    drift = coeffs[0] * (z[-1] - z[0])

    print(f"\n{fname}:")
    print(f"  Baseline slope  = {coeffs[0]:+.4f} V/V")
    print(f"  Baseline noise  = {noise:.5f} V")
    print(f"  Extrapolated drift over full Z = {drift:+.4f} V  "
          f"= {abs(drift/noise):.0f} sigma")
    print(f"  --> The baseline-deviation method sees this drift as ")
    print(f"      'contact' and triggers WAY too early!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (fname, fc) in zip(axes, curves.items()):
    z, d = fc.app_z_V, fc.app_defl_V
    n = len(z)
    bl_end = int(n * 0.10)
    coeffs = np.polyfit(z[:bl_end], d[:bl_end], 1)
    bl_pred = np.polyval(coeffs, z)
    noise = np.std(d[:bl_end] - bl_pred[:bl_end])
    deviation_sigma = np.abs(d - bl_pred) / noise

    ax.plot(100 * np.arange(n) / n, deviation_sigma, color="k", lw=0.4)
    ax.axhline(5, color="red", ls="--", lw=1, label="5 sigma threshold")
    ax.set_xlabel("Curve position (%)")
    ax.set_ylabel("Deviation from baseline fit (sigma)")
    ax.set_title(fname, fontsize=10)
    ax.legend(fontsize=8)
    ax.set_ylim(0, 30)
plt.suptitle("Baseline-deviation: gradual drift triggers false CP",
             fontweight="bold", y=1.02)
plt.tight_layout(); plt.show()
print("The deviation grows gradually due to baseline drift,")
print("not because of actual contact. A 5-sigma threshold")
print("triggers at 20-50% — far too early!")

## 3. Three CP Methods Compared

### Method 1: Piecewise-linear (classical)
Split curve into two lines, minimize total residual
$\mathcal{R}(k) = \text{SSR}_{\text{left}} + \text{SSR}_{\text{right}}$.
**Fails** due to length bias (the longer side dominates $\mathcal{R}$).

### Method 2: Baseline-deviation
Fit baseline, extrapolate, find first persistent deviation > $n_\sigma \cdot \sigma$.
**Fails** on sloped baselines — the extrapolation error grows linearly and
mimics contact.

### Method 3: Smoothed-gradient (what we use)
Compute the per-sample derivative of (smoothed) deflection:
$$g_i = \bar{d}_{i+1} - \bar{d}_i, \qquad \bar{d} = d * h_{\text{box}}$$

where $h_{\text{box}}$ is a box kernel of width $\approx N/60$.

On the **baseline**, $g$ is approximately constant (just the baseline slope expressed per sample). On **contact**, $g$ jumps sharply because the tip-surface interaction adds a steep slope on top of the baseline.

**Why it works:** the derivative cancels any *linear* baseline trend. A sloped baseline with slope $a$ contributes $g \approx a \cdot \Delta z$ to every sample — a constant offset, not a growing error. The CP shows up as an *abrupt change* in $g$, which is easy to threshold.

The CP is the first index where $g_i > \bar{g}_{\text{baseline}} + n_\sigma \cdot \sigma_g$ for $n_{\text{consec}}$ consecutive samples.

In [ ]:
results = {}
for fname, fc in curves.items():
    z, d = fc.app_z_V, fc.app_defl_V
    n = len(z)

    # Method 1: Piecewise-linear
    cp_pw = find_contact_piecewise(z, d)

    # Method 2: Baseline-deviation
    cp_bl = _baseline_deviation(z, d, direction="approach")

    # Method 3: Gradient (our method)
    cp_gr = estimate_contact_approach(z, d)

    results[fname] = (cp_pw, cp_bl, cp_gr)

    print(f"\n{fname}  (N = {n}):")
    print(f"  {'Method':<24s} | {'Index':>6s} | {'Pos %':>6s} | {'Defl (V)':>10s}")
    print(f"  {'-'*56}")
    for label, cp in [("Piecewise-linear", cp_pw),
                       ("Baseline-deviation", cp_bl),
                       ("Smoothed-gradient", cp_gr)]:
        print(f"  {label:<24s} | {cp.index:6d} | {100*cp.index/n:5.1f}% | "
              f"{cp.defl_V:+.5f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
colors = {"Piecewise": "red", "Baseline-dev": "orange", "Gradient": "green"}

for row, (fname, fc) in enumerate(curves.items()):
    z, d = fc.app_z_V, fc.app_defl_V
    n = len(z)
    cp_pw, cp_bl, cp_gr = results[fname]

    # Left: full curve
    ax = axes[row, 0]
    ax.plot(z, d, color=C_APP, lw=0.5, alpha=0.6)
    ax.axvline(z[cp_pw.index], color="red", ls="--", lw=1.2,
               label=f"Piecewise ({100*cp_pw.index/n:.0f}%)")
    ax.axvline(z[cp_bl.index], color="orange", ls="--", lw=1.2,
               label=f"Baseline-dev ({100*cp_bl.index/n:.0f}%)")
    ax.axvline(z[cp_gr.index], color="green", ls="-", lw=2,
               label=f"Gradient ({100*cp_gr.index/n:.0f}%)")
    ax.set_title(f"{fname} — full curve", fontsize=10)
    ax.set_xlabel("Z (V)"); ax.set_ylabel("Defl (V)")
    ax.legend(fontsize=7, loc="upper left")

    # Right: zoom on contact onset
    ax = axes[row, 1]
    margin = max(int(n * 0.05), 50)
    lo = max(0, cp_gr.index - margin)
    ax.plot(z[lo:], d[lo:], color=C_APP, lw=0.8, alpha=0.7)
    ax.axvline(z[cp_gr.index], color="green", ls="-", lw=2,
               label=f"Gradient CP (idx {cp_gr.index})")
    ax.set_title(f"Zoom: contact onset", fontsize=10)
    ax.set_xlabel("Z (V)"); ax.set_ylabel("Defl (V)")
    ax.legend(fontsize=8)

plt.tight_layout(); plt.show()

## 4. Visualising the Smoothed Gradient

The key insight: the **derivative** is flat on baseline and jumps at contact.
Let's plot it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (fname, fc) in zip(axes, curves.items()):
    d = fc.app_defl_V
    n = len(d)
    win = max(n // 60, 20)
    kernel = np.ones(win) / win
    d_smooth = np.convolve(d, kernel, mode='same')
    grad = np.diff(d_smooth)

    bl_end = int(n * 0.15)
    bl_mean = np.mean(grad[:bl_end])
    bl_std = np.std(grad[:bl_end])

    pct = 100 * np.arange(len(grad)) / n
    sigma = (grad - bl_mean) / bl_std

    ax.plot(pct, sigma, color="k", lw=0.4)
    ax.axhline(5, color="green", ls="--", lw=1, label="5 sigma threshold")
    ax.axhline(0, color="grey", lw=0.5)

    cp = estimate_contact_approach(fc.app_z_V, d)
    ax.axvline(100*cp.index/n, color="green", lw=2, alpha=0.7,
               label=f"CP at {100*cp.index/n:.1f}%")

    ax.set_xlabel("Curve position (%)")
    ax.set_ylabel("Gradient (sigma above baseline)")
    ax.set_title(fname, fontsize=10)
    ax.set_ylim(-10, 50)
    ax.legend(fontsize=8)

plt.suptitle("Smoothed gradient: flat on baseline, sharp jump at contact",
             fontweight="bold", y=1.02)
plt.tight_layout(); plt.show()

## 5. Retract Contact Point

For retract we walk *backward* from baseline and check the gradient
**magnitude** (the sign depends on whether there is adhesion).

In [ ]:
for fname, fc in curves.items():
    cp_r = estimate_contact_retract(fc.ret_z_V, fc.ret_defl_V)
    n_r = len(fc.ret_z_V)
    print(f"{fname}: retract CP at idx={cp_r.index} ({100*cp_r.index/n_r:.1f}%), "
          f"defl={cp_r.defl_V:+.5f}")

## 6. Cantilever Linearity & INVOLS

On a stiff sample, $\partial d / \partial z = \text{const}$ in contact.
In practice the local slope drifts with indentation. We slide a small
window from the CP outward, compute the local slope at each position,
and define the **linear region** as the range where the slope stays
within $\pm\theta\%$ of the near-CP reference.

$$\text{INVOLS} = \frac{z_{\text{scale}} \times 1000}{|s_{\text{linear}}|} \quad (\text{nm/V})$$

In [ ]:
z_scale = 30.0  # um/V — adjust to your setup

for fname, fc in curves.items():
    cp = estimate_contact_approach(fc.app_z_V, fc.app_defl_V)
    contact_z = fc.app_z_V[cp.index:]
    contact_d = fc.app_defl_V[cp.index:]
    n_c = len(contact_z)
    z_cp = contact_z[0]

    win = max(int(n_c * 0.04), 20)
    step = max(1, (n_c - win) // 200)
    dz_arr, slopes = [], []
    for i in range(0, n_c - win, step):
        coeffs = np.polyfit(contact_z[i:i+win], contact_d[i:i+win], 1)
        mid = i + win // 2
        dz_arr.append((contact_z[mid] - z_cp) * z_scale * 1000)
        slopes.append(coeffs[0])
    dz_arr = np.array(dz_arr)
    slopes = np.array(slopes)

    skip = max(2, len(slopes)//50)
    ref_end = skip + max(len(slopes)//8, 5)
    ref_slope = np.median(slopes[skip:ref_end])
    dev = 100 * (slopes - ref_slope) / abs(ref_slope)

    # Find linear end at 10%
    thresh = 10.0
    end_idx = len(dev) - 1
    for i in range(ref_end, len(dev) - 4):
        if all(abs(dev[i+j]) > thresh for j in range(4)):
            end_idx = i; break

    invols = abs(z_scale * 1000 / ref_slope) if abs(ref_slope) > 1e-12 else 0

    print(f"\n{fname}:")
    print(f"  Contact region: {n_c} pts, {dz_arr[-1]:.0f} nm range")
    print(f"  Reference slope: {ref_slope:.2f} V/V")
    print(f"  Linear range: {dz_arr[end_idx]:.0f} nm (10% threshold)")
    print(f"  INVOLS (linear region): {invols:.1f} nm/V")

## 7. Summary

| Method | Principle | Failure mode |
|---|---|---|
| Piecewise-linear | Minimize total two-line residual | Length bias: long contact side dominates |
| Baseline-deviation | Extrapolate baseline fit | Sloped baseline: drift mimics contact |
| **Smoothed-gradient** | Detect jump in derivative | **Robust**: derivative cancels linear drift |

The smoothed-gradient method works because the *derivative* of a linearly-drifting
baseline is constant — only an actual slope change (contact) produces a detectable
jump.

### Retract
The gradient magnitude is checked walking backward from baseline, correctly
identifying the snap-back / separation point regardless of adhesion dips.

### Linearity
The cantilever's linear region is determined by a sliding-window slope analysis
from the CP outward. INVOLS should be computed from this region only, not the
full contact zone.